In [ ]:
# AVG / CLS 両対応 特徴ベクトル抽出コード
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
from pathlib import Path

import torch
from PIL import Image
import timm
from timm.data import resolve_data_config, create_transform


In [ ]:

# =========================
# 設定
# =========================
EXPERIMENTS = [
    # ======================================================
    # 1. 実証済み ViT CLS
    # ======================================================
    {
        "exp_name": "vit_large_patch16_384_augreg_CLS_384",
        "model_name": "vit_large_patch16_384.augreg_in21k_ft_in1k",
        "img_size": 384,
        "feature_type": "cls",
    },
    {
        "exp_name": "vit_large_patch16_384_augreg_CLS_616",
        "model_name": "vit_large_patch16_384.augreg_in21k_ft_in1k",
        "img_size": 616,
        "feature_type": "cls",
    },
    {
        "exp_name": "vit_large_patch16_384_augreg_AVG_384",
        "model_name": "vit_large_patch16_384.augreg_in21k_ft_in1k",
        "img_size": 384,
        "feature_type": "avg",
    },
    {
        "exp_name": "vit_large_patch16_384_augreg_AVG_616",
        "model_name": "vit_large_patch16_384.augreg_in21k_ft_in1k",
        "img_size": 616,
        "feature_type": "avg",
    },


    # ======================================================
    # 2. EVA02 AVG
    # ======================================================
    {
        "exp_name": "eva02_large_patch14_448_mim_in22k_ft_in22k_AVG_448",
        "model_name": "eva02_large_patch14_448.mim_in22k_ft_in22k",
        "img_size": 448,
        "feature_type": "avg",
    },
    {
        "exp_name": "eva02_large_patch14_448_mim_in22k_ft_in22k_in1k_AVG_448",
        "model_name": "eva02_large_patch14_448.mim_in22k_ft_in22k_in1k",
        "img_size": 448,
        "feature_type": "avg",
    },
    {
        "exp_name": "eva02_large_patch14_448_mim_m38m_ft_in22k_AVG_448",
        "model_name": "eva02_large_patch14_448.mim_m38m_ft_in22k",
        "img_size": 448,
        "feature_type": "avg",
    },
    {
        "exp_name": "eva02_large_patch14_448_mim_m38m_ft_in22k_in1k_AVG_448",
        "model_name": "eva02_large_patch14_448.mim_m38m_ft_in22k_in1k",
        "img_size": 448,
        "feature_type": "avg",
    },

    # ======================================================
    # 3. DINO / SigLIP
    # ======================================================
    {
        "exp_name": "vit_large_patch14_dinov2_AVG_518",
        "model_name": "vit_large_patch14_dinov2.lvd142m",
        "img_size": 518,
        "feature_type": "avg",
    },
    {
        "exp_name": "vit_large_patch16_siglip_gap_512_AVG_512",
        "model_name": "vit_large_patch16_siglip_gap_512.v2_webli",
        "img_size": 512,
        "feature_type": "avg",
    },

    # ======================================================
    # 4. ConvNet / CNN系
    # ======================================================
    {
        "exp_name": "tf_efficientnetv2_l_AVG_480",
        "model_name": "tf_efficientnetv2_l.in21k_ft_in1k",
        "img_size": 480,
        "feature_type": "avg",
    },
    {
        "exp_name": "convnextv2_large_fcmae_AVG_384",
        "model_name": "convnextv2_large.fcmae_ft_in22k_in1k_384",
        "img_size": 384,
        "feature_type": "avg",
    },

    # ======================================================
    # 5. Swin / Hybrid / MetaFormer
    # ======================================================
    {
        "exp_name": "swinv2_large_window12to24_AVG_384",
        "model_name": "swinv2_large_window12to24_192to384.ms_in22k_ft_in1k",
        "img_size": 384,
        "feature_type": "avg",
    },
    {
        "exp_name": "caformer_b36_AVG_384",
        "model_name": "caformer_b36.sail_in22k_ft_in1k_384",
        "img_size": 384,
        "feature_type": "avg",
    },
    {
        "exp_name": "maxxvitv2_rmlp_base_AVG_384",
        "model_name": "maxxvitv2_rmlp_base_rw_384.sw_in12k_ft_in1k",
        "img_size": 384,
        "feature_type": "avg",
    },
]
EXPERIMENTS = [
    # ======================================================
    # 実証済み ViT CLS
    # ======================================================
    {
        "exp_name": "vit_large_patch16_384_augreg_CLS_default384",
        "model_name": "vit_large_patch16_384.augreg_in21k_ft_in1k",
        "feature_type": "cls",
        "img_size": None,   # モデル既定値 384
    },
    {
        "exp_name": "vit_large_patch16_384_augreg_CLS_616",
        "model_name": "vit_large_patch16_384.augreg_in21k_ft_in1k",
        "feature_type": "cls",
        "img_size": 616,    # 先生の実証済み条件
    },

    # ======================================================
    # EVA02 AVG
    # ======================================================
    {
        "exp_name": "eva02_large_patch14_448_mim_in22k_ft_in22k_in1k_AVG",
        "model_name": "eva02_large_patch14_448.mim_in22k_ft_in22k_in1k",
        "feature_type": "avg",
        "img_size": None,   # モデル既定値 448
    },
    {
        "exp_name": "eva02_large_patch14_448_mim_in22k_ft_in22k_AVG",
        "model_name": "eva02_large_patch14_448.mim_in22k_ft_in22k",
        "feature_type": "avg",
        "img_size": None,   # モデル既定値 448
    },

    # ======================================================
    # 追加候補
    # ======================================================
    # {
    #     "exp_name": "vit_large_patch14_dinov2_AVG",
    #     "model_name": "vit_large_patch14_dinov2.lvd142m",
    #     "feature_type": "avg",
    #     "img_size": None,
    # },
    # {
    #     "exp_name": "swinv2_large_window12to24_AVG",
    #     "model_name": "swinv2_large_window12to24_192to384.ms_in22k_ft_in1k",
    #     "feature_type": "avg",
    #     "img_size": None,
    # },
    # {
    #     "exp_name": "convnextv2_large_AVG",
    #     "model_name": "convnextv2_large.fcmae_ft_in22k_in1k_384",
    #     "feature_type": "avg",
    #     "img_size": None,
    # },
]


In [ ]:

INPUT_FOLDER = Path("/home/tatsushi/デスクトップ/sensitivity_data7/image-aug")
OUTPUT_ROOT = Path("/home/tatsushi/デスクトップ/sensitivity_data7/aug-feature")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VALID_EXT = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp"}


# =========================
# 画像パス列挙
# =========================
def list_images_once(input_dir: Path):
    image_paths = []
    for root, _, files in os.walk(input_dir):
        root = Path(root)
        for fname in files:
            if Path(fname).suffix.lower() in VALID_EXT:
                image_paths.append(root / fname)
    image_paths.sort()
    return image_paths


# =========================
# ユーティリティ
# =========================
def clean_model_name(name: str) -> str:
    return name.strip().rstrip("#").strip()


def make_safe_name(name: str) -> str:
    return (
        name.replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace("#", "")
        .strip()
    )


def create_model_for_feature(model_name: str, feature_type: str, img_size=None):
    """
    feature_type:
        "avg" : timm の global_pool="avg" を使う
        "cls" : forward_features から CLS token を取り出す
    """

    model_name = clean_model_name(model_name)

    if feature_type not in ["avg", "cls"]:
        raise ValueError("feature_type は 'avg' または 'cls' にしてください。")

    # AVGの場合は、timm側のpoolingを使う
    if feature_type == "avg":
        create_kwargs = {
            "pretrained": True,
            "num_classes": 0,
            "global_pool": "avg",
        }

    # CLSの場合は、poolingせず forward_features からCLSを取る
    else:
        create_kwargs = {
            "pretrained": True,
            "num_classes": 0,
            "global_pool": "",
        }

    # ViT CLS 616など、既定値以外を使う場合だけ img_size を渡す
    if img_size is not None:
        create_kwargs["img_size"] = img_size

        # timmのViT系で可変サイズ入力を安定させるため
        # モデルによっては dynamic_img_size を受け取らない場合があるので try で処理する
        create_kwargs["dynamic_img_size"] = True

    try:
        model = timm.create_model(model_name, **create_kwargs)

    except TypeError:
        # dynamic_img_size を受け取れないモデル用
        create_kwargs.pop("dynamic_img_size", None)
        model = timm.create_model(model_name, **create_kwargs)

    return model


def create_transform_for_model(model, img_size=None):
    """
    img_size=None のときはモデル既定値。
    img_size指定時のみ input_size を強制する。
    """

    if img_size is None:
        data_cfg = resolve_data_config({}, model=model)
    else:
        data_cfg = resolve_data_config(
            {"input_size": (3, img_size, img_size)},
            model=model,
        )

    transform = create_transform(**data_cfg)
    return transform, data_cfg


def extract_feature(model, x, feature_type: str):
    """
    feature_type:
        avg : model(x) で timm の global_pool="avg" 出力を使う
        cls : forward_features(x) から CLS token を取り出す
    """

    if feature_type == "avg":
        feat = model(x)
        return feat

    # CLSの場合
    if not hasattr(model, "forward_features"):
        raise RuntimeError("このモデルは forward_features を持たないため CLS 抽出できません。")

    feats = model.forward_features(x)

    # timmの一部モデルがdictを返す場合への保険
    if isinstance(feats, dict):
        for key in ["x_norm_clstoken", "cls_token", "pooled", "features"]:
            if key in feats:
                feats = feats[key]
                break
        else:
            raise RuntimeError(f"dict形式のforward_featuresに対応できません: keys={feats.keys()}")

    # ViT系: [B, tokens, dim]
    if feats.ndim == 3:
        return feats[:, 0]  # CLS token

    # CNN系など: [B, C, H, W]
    # CLS指定でここに来るのは通常想定外だが、保険としてGAP
    if feats.ndim == 4:
        return feats.mean(dim=(2, 3))

    # すでに [B, dim] の場合
    if feats.ndim == 2:
        return feats

    raise RuntimeError(f"想定外の特徴形状です: {feats.shape}")


# =========================
# main
# =========================
def main():
    print(f"DEVICE: {DEVICE}")

    image_paths = list_images_once(INPUT_FOLDER)
    print(f"Found {len(image_paths)} images under: {INPUT_FOLDER}")

    if len(image_paths) == 0:
        print("画像が見つかりません。INPUT_FOLDERを確認してください。")
        return

    for exp in EXPERIMENTS:
        exp_name = make_safe_name(exp["exp_name"])
        model_name = clean_model_name(exp["model_name"])
        feature_type = exp["feature_type"].lower()
        img_size = exp.get("img_size", None)

        print("\n" + "=" * 80)
        print(f"Experiment : {exp_name}")
        print(f"Model      : {model_name}")
        print(f"Feature    : {feature_type.upper()}")
        print(f"Img size   : {img_size if img_size is not None else 'model default'}")
        print("=" * 80)

        # モデル作成
        model = create_model_for_feature(
            model_name=model_name,
            feature_type=feature_type,
            img_size=img_size,
        )

        model.to(DEVICE).eval()
        for p in model.parameters():
            p.requires_grad = False

        # transform作成
        transform, data_cfg = create_transform_for_model(model, img_size=img_size)
        print(f"Transform input_size: {data_cfg.get('input_size')}")

        # 出力フォルダ
        out_model_root = OUTPUT_ROOT / exp_name
        out_model_root.mkdir(parents=True, exist_ok=True)

        # 特徴抽出
        saved_count = 0
        skip_count = 0
        error_count = 0

        with torch.inference_mode():
            for idx, in_path in enumerate(image_paths, start=1):
                rel_dir = in_path.parent.relative_to(INPUT_FOLDER)
                out_dir = out_model_root / rel_dir
                out_dir.mkdir(parents=True, exist_ok=True)

                out_path = out_dir / f"{in_path.stem}_feature.pt"

                if out_path.exists():
                    skip_count += 1
                    continue

                try:
                    img = Image.open(in_path).convert("RGB")
                    x = transform(img).unsqueeze(0).to(DEVICE)

                    feat = extract_feature(
                        model=model,
                        x=x,
                        feature_type=feature_type,
                    )

                    feat = feat.squeeze(0).contiguous().cpu()
                    torch.save(feat, out_path)
                    saved_count += 1

                except Exception as e:
                    error_count += 1
                    print(f"[ERROR] {in_path}")
                    print(f"        {type(e).__name__}: {e}")
                    continue

                if idx % 100 == 0:
                    print(f"Processed {idx}/{len(image_paths)}")

        print(f"Saved features to: {out_model_root}")
        print(f"Saved: {saved_count}, Skipped: {skip_count}, Errors: {error_count}")

        # GPUメモリ解放
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print("\nDone.")


if __name__ == "__main__":
    main()